In [27]:
import pandas as pd
import numpy as np

In [ ]:
import numpy as np
import pandas as pd

def perform_reconciliation(tb_df,gl_df):
    rec_df=gl_df.merge(tb_df,on='Account',how="outer",suffixes=('_GL','_TB'))

    rec_df['Difference']=(rec_df['Amount_GL']).fillna(0)-(rec_df['Amount_TB']).fillna(0)

    raw_variance=(rec_df['Difference']/rec_df['Amount_TB'])
    is_invalid_tb = rec_df['Amount_TB'].isna() | (rec_df['Amount_TB'] == 0) | np.isinf(raw_variance)
    rec_df['Variance'] = np.where(
        is_invalid_tb, 
        "N/A", 
        raw_variance.map(lambda x: f"{x*100:.2f}%" if pd.notna(x) else "N/A")
    )

    conditions=[
        rec_df['Amount_GL'].isna(),
        rec_df['Amount_TB'].isna(),
        rec_df['Difference']==0
    ]

    choices=[
        "Missing in GL",
        "Missing in TB",
        "Matched"
    ]

    rec_df['Status']=np.select(conditions,choices,default="Mismatch")
    summary=rec_df['Status'].value_counts().to_dict()
    return rec_df,summary

In [38]:
tb_data = {
    'Account': ['1010', '1020', '1030', '1040'],
    'Amount_TB': [5000.0, 2500.0, 1200.0, 4000.0]
}
tb_df = pd.DataFrame(tb_data)
gl_data = {
    'Account': ['1010', '1020', '1050'],
    'Amount_GL': [5000.0, 2600.0, 800.0]
}
gl_df = pd.DataFrame(gl_data)

In [39]:
rec_df,summary=perform_reconciliation(tb_df,gl_df)

In [40]:

print("--- RECONCILIATION DATAFRAME ---")
print(rec_df)
print("\n--- SUMMARY DICTIONARY (Value Counts of Differences) ---")
print(summary)

--- RECONCILIATION DATAFRAME ---
  Account  Amount_GL  Amount_TB  Difference Variance         Status
0    1010     5000.0     5000.0         0.0     0.0%        Matched
1    1020     2600.0     2500.0       100.0     4.0%       Mismatch
2    1030        NaN     1200.0         0.0     0.0%  Missing in GL
3    1040        NaN     4000.0         0.0     0.0%  Missing in GL
4    1050      800.0        NaN         0.0     0.0%  Missing in TB

--- SUMMARY DICTIONARY (Value Counts of Differences) ---
{'Missing in GL': 2, 'Matched': 1, 'Mismatch': 1, 'Missing in TB': 1}


In [47]:
def calculate_kpis(rec_df,summary):
    val_dict={
        "Total Accounts":rec_df.shape[0],
        "Match Rate":f"{((((rec_df[rec_df['Status']=='Matched']).shape[0]))/rec_df.shape[0])*100:.2f}%",
        "Total GL":round(float((rec_df['Amount_GL']).sum()),2),
        "Total TB":round(float((rec_df['Amount_TB']).sum()),2),
        "Net Difference":round(float(((rec_df['Amount_GL']).sum())-((rec_df['Amount_TB']).sum())),2),
        "Largest Variance Account":rec_df.loc[rec_df['Difference'].abs().idxmax,'Account'],
        "Largest Variance Amount":float(rec_df.loc[rec_df['Difference'].abs().idxmax,"Difference"])
    }
    kpi_summary=val_dict | summary
    return kpi_summary

In [45]:
kpi_sum=calculate_kpis(rec_df,summary)

In [46]:
print(kpi_sum)

{'Total Accounts': 5, 'Match Rate': '20.00%', 'Total GL': 8400.0, 'Total TB': 12700.0, 'Net Difference': -4300.0, 'Missing in GL': 2, 'Matched': 1, 'Mismatch': 1, 'Missing in TB': 1}
